In [ ]:
# ============================================================
# Cell 1: 初始化
# ============================================================
setwd("/home/ailab/caohao/AdaDiss/")

suppressPackageStartupMessages({
    library(Seurat)
    library(harmony)
    library(dplyr)
    library(ggplot2)
    library(Matrix)
})

paths <- list(
    raw = list(
        scrna_dir  = "./data/raw/scrna/GSE264393_RAW/",
        xenium_dir = "./data/raw/xenium/GSE264334_RAW/",
        xenium_tx  = "./data/raw/xenium/GSE264334_RAW/transcripts.csv.gz",
        xenium_cells = "./data/raw/xenium/GSE264334_RAW/cells.csv.gz"
    ),
    cache = list(
        scrna  = "./data/cache/kidney_scrna_processed.rds",
        xenium = "./data/cache/kidney_xenium_processed.rds"
    )
)
for (d in c("./data/cache", "./results", "./plots")) {
    dir.create(d, showWarnings = FALSE, recursive = TRUE)
}

# 论文 Methods 过滤参数（人肾脏 snRNA-seq）
params <- list(
    min_features  = 500,
    max_features  = 6000,
    max_counts    = 25000,
    max_mt_pct    = 25,
    n_pcs         = 30,
    cluster_res   = 0.5,
    harmony_theta = 2
)
SAMPLES <- c("A1", "C3", "C4", "C5")
cat("初始化完成\n")

In [ ]:
# ============================================================
# Cell 2: 加载 4 个 snRNA-seq 样本并合并
# GSE264393_RAW 文件结构: {sample}_barcodes/features/matrix.tsv/mtx.gz
# ============================================================

if (file.exists(paths$cache$scrna)) {
    cat("[Cache HIT] 加载 scRNA 缓存...\n")
    scrna_obj <- readRDS(paths$cache$scrna)
    cat("细胞数:", ncol(scrna_obj), "\n")
    cat("细胞类型:", paste(sort(unique(scrna_obj$cell_type)), collapse = ", "), "\n")
} else {
    cat("[Cache MISS] 从原始数据构建 scRNA 对象...\n")

    obj_list <- list()
    for (sample in SAMPLES) {
        cat("  加载样本", sample, "...\n")
        tmp <- file.path(tempdir(), sample)
        dir.create(tmp, showWarnings = FALSE)
        file.copy(file.path(paths$raw$scrna_dir,
                            paste0(sample, "_barcodes.tsv.gz")),
                  file.path(tmp, "barcodes.tsv.gz"))
        file.copy(file.path(paths$raw$scrna_dir,
                            paste0(sample, "_features.tsv.gz")),
                  file.path(tmp, "features.tsv.gz"))
        file.copy(file.path(paths$raw$scrna_dir,
                            paste0(sample, "_matrix.mtx.gz")),
                  file.path(tmp, "matrix.mtx.gz"))
        obj <- CreateSeuratObject(
            counts = Read10X(data.dir = tmp),
            project = sample, min.cells = 3, min.features = 200
        )
        obj$sample <- sample
        obj_list[[sample]] <- obj
        cat("    核数:", ncol(obj), "\n")
    }
    scrna_raw <- merge(obj_list[[1]], y = obj_list[-1],
                       add.cell.ids = SAMPLES)
    cat("合并后总核数:", ncol(scrna_raw), "\n")

    # QC 过滤（论文参数）
    scrna_raw[["percent.mt"]] <- PercentageFeatureSet(scrna_raw, pattern = "^MT-")
    scrna_filt <- subset(scrna_raw,
        subset = nFeature_RNA > params$min_features &
                 nFeature_RNA < params$max_features &
                 nCount_RNA   < params$max_counts   &
                 percent.mt   < params$max_mt_pct)
    cat("QC 过滤后:", ncol(scrna_filt), "个核\n")

    # 标准化 + PCA + Harmony + UMAP + 聚类
    cat("标准化和降维...\n")
    scrna_obj <- scrna_filt                                  %>%
        NormalizeData(verbose = FALSE)                       %>%
        FindVariableFeatures(nfeatures = 3000, verbose = FALSE) %>%
        ScaleData(verbose = FALSE)                           %>%
        RunPCA(npcs = params$n_pcs, verbose = FALSE)

    cat("Harmony 批次校正...\n")
    scrna_obj <- RunHarmony(scrna_obj,
        group.by.vars = "sample",
        theta = params$harmony_theta, verbose = FALSE)

    cat("UMAP + 聚类 (res=", params$cluster_res, ")...\n")
    scrna_obj <- scrna_obj                                   %>%
        RunUMAP(reduction = "harmony",
                dims = 1:params$n_pcs, verbose = FALSE)      %>%
        FindNeighbors(reduction = "harmony",
                      dims = 1:params$n_pcs, verbose = FALSE) %>%
        FindClusters(resolution = params$cluster_res,
                     verbose = FALSE)
    cat("聚类数:", length(unique(scrna_obj$seurat_clusters)), "\n")
}

# 可视化（无论从缓存还是新建都执行）
DimPlot(scrna_obj,
        group.by = ifelse("cell_type" %in% colnames(scrna_obj@meta.data),
                          "cell_type", "seurat_clusters"),
        reduction = "umap", label = TRUE, repel = TRUE, pt.size = 0.3)

In [ ]:
# ============================================================
# Cell 3: 查看 Marker 基因（仅首次运行需要，用于手动注释 Cell 4）
# 如果已从缓存加载且有 cell_type，可跳过此 Cell
# ============================================================

if (!"cell_type" %in% colnames(scrna_obj@meta.data)) {
    cat("计算 marker genes（需要几分钟）...\n")
    markers <- FindAllMarkers(
        scrna_obj, only.pos = TRUE,
        min.pct = 0.25, logfc.threshold = 0.25, verbose = FALSE
    )
    top5 <- markers %>%
        group_by(cluster) %>%
        slice_max(avg_log2FC, n = 5) %>%
        select(cluster, gene, avg_log2FC, pct.1)
    print(top5, n = Inf)

    # 肾脏细胞类型标志基因（参考 TopACT 论文 Extended Data Fig.1）
    kidney_markers <- list(
        PCT   = c("CUBN", "LRP2", "SLC5A2"),
        TAL   = c("SLC12A1", "UMOD", "CLDN16"),
        LOH   = c("SLC12A1", "AQP1"),
        DCT   = c("SLC12A3", "TRPM6", "CALB1"),
        PC    = c("AQP2", "AQP3", "FXYD4"),
        `IC-A`= c("SLC4A1", "ATP6V1B1"),
        `IC-B`= c("SLC26A4"),
        ENDO  = c("PECAM1", "FLT1", "EMCN"),
        MES   = c("PDGFRB", "ITGA8"),
        POD   = c("NPHS1", "NPHS2", "SYNPO"),
        PEC   = c("PAX8", "CLDN1"),
        LEU   = c("PTPRC", "CD74")
    )
    marker_genes_flat <- unique(unlist(lapply(kidney_markers, head, 2)))
    marker_genes_flat <- marker_genes_flat[
        marker_genes_flat %in% rownames(scrna_obj)]

    print(DotPlot(scrna_obj, features = marker_genes_flat,
                  group.by = "seurat_clusters") +
          RotatedAxis() +
          ggtitle("Marker Expression — 对照此图填写 Cell 4 的映射"))
} else {
    cat("已有 cell_type 注释，跳过 marker 计算\n")
}

In [ ]:
# ============================================================
# Cell 4: 手动注释 cluster → 细胞类型
# ⚠️ 根据 Cell 3 的 DotPlot 结果修改下面的映射
# 如果从缓存加载已有注释则跳过
# ============================================================

if (!"cell_type" %in% colnames(scrna_obj@meta.data)) {

    # ── 根据 Cell 3 DotPlot 填写 ──────────────────────────────
    cluster_to_celltype <- c(
        "0"  = "PCT",
        "1"  = "PCT",
        "2"  = "PCT",
        "3"  = "TAL",
        "4"  = "TAL",
        "5"  = "DCT",
        "6"  = "ENDO",
        "7"  = "PC",
        "8"  = "LOH",
        "9"  = "PC",
        "10" = "MES",
        "11" = "IC-A",
        "12" = "POD",
        "13" = "PCT",
        "14" = "TAL",
        "15" = "PEC",
        "16" = "IC-B",
        "17" = "ENDO",
        "18" = "LOH",
        "19" = "LEU",
        "20" = "PCT",
        "21" = "PC",
        "22" = "MES",
        "23" = "LEU"
        # cluster 数量由 Cell 2 决定，如不够请继续添加
    )

    scrna_obj$cell_type <- cluster_to_celltype[
        as.character(scrna_obj$seurat_clusters)]

    na_n <- sum(is.na(scrna_obj$cell_type))
    if (na_n > 0) {
        miss_cls <- unique(scrna_obj$seurat_clusters[
            is.na(scrna_obj$cell_type)])
        warning(na_n, " 个细胞未映射，cluster: ",
                paste(miss_cls, collapse = ", "),
                "\n请在上方映射表中补充这些 cluster")
        scrna_obj$cell_type[is.na(scrna_obj$cell_type)] <- "Unknown"
    }

    cat("细胞类型分布：\n")
    print(table(scrna_obj$cell_type))

    # 保存缓存（含注释）
    saveRDS(scrna_obj, paths$cache$scrna)
    cat("\n已保存 scRNA 缓存:", paths$cache$scrna, "\n")
}

DimPlot(scrna_obj, group.by = "cell_type",
        reduction = "umap", label = TRUE, repel = TRUE, pt.size = 0.3) +
    ggtitle("Annotated Cell Types — Healthy Human Kidney snRNA-seq")

In [ ]:
# ============================================================
# Cell 5: 加载 Xenium IgAN 肾脏数据（GSE264334_RAW）
# 列名已确认：cell_id, x_centroid, y_centroid
# ============================================================

cat("加载 Xenium IgAN 肾脏数据...\n")

# ── 加载 cell-level counts（10x 格式）────────────────────────
# GSE264334_RAW 含 barcodes/features/matrix，可直接 Read10X
xenium_counts <- Read10X(data.dir = paths$raw$xenium_dir)
xenium_obj    <- CreateSeuratObject(
    counts = xenium_counts, project = "Kidney_IgAN", min.cells = 3
)
cat("Xenium 细胞数:", ncol(xenium_obj), "\n")
cat("Xenium 基因数:", nrow(xenium_obj), "\n")

# ── 加载细胞质心坐标（列名已确认）────────────────────────────
# 确认列名：cell_id, x_centroid, y_centroid
cells_meta <- read.csv(
    gzcon(file(paths$raw$xenium_cells, "rb"))
)
cat("cells.csv.gz 列名:", paste(colnames(cells_meta), collapse = ", "), "\n")

# 以 cell_id 作为行名，与 Seurat barcode 对齐
rownames(cells_meta) <- cells_meta$cell_id
common_cells <- intersect(colnames(xenium_obj), rownames(cells_meta))
cat("匹配细胞数:", length(common_cells), "\n")

xenium_obj <- xenium_obj[, common_cells]
xenium_obj <- AddMetaData(
    xenium_obj,
    metadata = cells_meta[common_cells,
        c("x_centroid", "y_centroid", "transcript_counts",
          "cell_area", "nucleus_area")]
)
cat("坐标已加载 — x_centroid 范围:",
    round(min(xenium_obj$x_centroid), 1), "~",
    round(max(xenium_obj$x_centroid), 1), "μm\n")

In [ ]:
# ============================================================
# Cell 6: Seurat 标签转移（定性参考，不参与 GNN 训练）
# ============================================================

cat("Seurat 标签转移（仅作定性对比）...\n")

common_genes <- intersect(rownames(scrna_obj), rownames(xenium_obj))
cat("共同基因数:", length(common_genes), "\n")

scrna_sub  <- scrna_obj[common_genes, ]
xenium_sub <- xenium_obj[common_genes, ]
DefaultAssay(xenium_sub) <- "RNA"

cat("寻找锚点...\n")
anchors <- FindTransferAnchors(
    reference            = scrna_sub,
    query                = xenium_sub,
    normalization.method = "LogNormalize",
    reference.reduction  = "harmony",
    dims                 = 1:params$n_pcs
)

cat("转移标签...\n")
predictions <- TransferData(
    anchorset = anchors,
    refdata   = scrna_obj$cell_type,
    dims      = 1:params$n_pcs
)
xenium_obj <- AddMetaData(xenium_obj, metadata = predictions)

cat("\nSeurat 标签转移结果分布（仅定性参考，不参与模型选择）：\n")
print(table(xenium_obj$predicted.id))

In [ ]:
# ============================================================
# Cell 7: 保存缓存
# ============================================================

saveRDS(scrna_obj,  paths$cache$scrna)
saveRDS(xenium_obj, paths$cache$xenium)

cat("=== 保存完成 ===", "\n")
cat("scRNA 缓存  :", paths$cache$scrna, "\n")
cat("Xenium 缓存 :", paths$cache$xenium, "\n")
cat("\n下一步：运行 train_copy_spot_level.ipynb\n")
cat("  将从 transcripts.csv.gz 构建 spot 图（不依赖细胞分割）\n")